# Build MDE webapp data (`docs/data/mde.json`)

Exports the canonical Replogle-pipeline-modified MDE (coords + leiden/hdbscan cluster ids) plus manually-curated cluster labels and per-pert n_DEGs as a compact JSON the static viewer (`docs/`) consumes.

**Inputs**
- `KOLF_Perturbation_Atlas_Analysis/output_files/KOLF_Pan_Genome_Filtered_Replogle_Pipeline_Modified_MDE_clusters_annotated.xlsx` — sheet `MDE` has 1,656 perts with x, y, gene_target, leiden, hdbscan. Produced by `psp.da.plot_mde` → `psp.da.annotate_clusters` in `KOLF_Perturbation_Atlas_Data_Analysis.ipynb`.
- `MANUAL_LEIDEN` / `MANUAL_HDBSCAN` (below) — 62 and 30 manually curated labels indexed by cluster id (empty string = no manual label; viewer will show just `cluster N`).
- `psp/notebooks/input_files/n_degs_k562_kolf_rpe1.csv` — per-pert n_DEGs across cell types (we take the `KOLF` column).

**Output**
- `docs/data/mde.json`:
    ```
    {
      "points": [{"g", "x", "y", "l", "h", "n"}, ...],
      "leiden_labels":  {<cluster_id>: <manual label or "">},
      "hdbscan_labels": {<cluster_id>: <manual label or "">}
    }
    ```

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('/tscc/projects/ps-malilab/ydoctor/KOLF_Perturbation_Atlas')
XLSX  = ROOT / 'KOLF_Perturbation_Atlas_Analysis/output_files/KOLF_Pan_Genome_Filtered_Replogle_Pipeline_Modified_MDE_clusters_annotated.xlsx'
NDEGS = ROOT / 'psp/notebooks/input_files/n_degs_k562_kolf_rpe1.csv'
OUT   = ROOT / 'docs/data/mde.json'

In [ ]:
# Manual leiden cluster labels, indexed 0..61.
MANUAL_LEIDEN = [
    "", "", "", "", "",
    "npBAF complex; Cohesin complex; PcG complex",
    "",
    "Transcription Regulation",
    "Mitochondrial Translation",
    "",
    "Mitochondrial Translation",
    "",
    "DNA Repair",
    "",
    "Mitochondrial Translation",
    "Neddylation",
    "",
    "P-Body; NuA4 Complex",
    "SAGA complex; Prefoldin-like complex; Mediator complex",
    "",
    "Oxidative Phosphorylation",
    "",
    "Mitochondrial RNA metabolic process",
    "",
    "U2 snRNP",
    "Ragulator Complex; Respirasome",
    "",
    "Lysosome",
    "",
    "tRNA Aminoacylation",
    "",
    "Mitochondrial Membrane",
    "TFIID complex",
    "Pluripotency Signaling",
    "",
    "Cortical Cytoskeleton",
    "m6A Transferase; INO80 complex; CREBBP/EP300",
    "Transcription Regulation",
    "Mitochondrial Membrane; Apoptosome",
    "snRNA Transcription; Cajal Body; SMN complex",
    "SIN3 Complex; Histone Methyltransferase",
    "Translation Initiation",
    "",
    "Endoplasmic Reticulum",
    "Membrane Trafficking",
    "Mitochondrial Translation",
    "Ubiquitin Complex; Methyltransferase complex",
    "AP-2 adaptor complex",
    "tRNA processing",
    "",
    "",
    "POU domain",
    "",
    "",
    "DNA methylation",
    "Methylosome",
    "",
    "Centromeric Region",
    "Protein Ufmylation",
    "TGF-beta Signaling",
    "",
    "Mitochondrial Translation",
]
assert len(MANUAL_LEIDEN) == 62, f'expected 62 leiden labels, got {len(MANUAL_LEIDEN)}'

In [ ]:
# Manual HDBSCAN cluster labels, indexed 0..29. Noise (cluster -1) is hard-coded
# to 'unclustered' in the JS viewer, not in this list.
MANUAL_HDBSCAN = [
    "AP2 complex",                                    # 0
    "RNA Pol II complex",                             # 1
    "Cortical Cytoskeleton",                          # 2
    "Cohesin complex",                                # 3
    "npBAF complex; PcG complex",                     # 4
    "INO80 complex",                                  # 5
    "Hippo-YAP Signaling",                            # 6
    "Heparan Sulfate Proteoglycan",                   # 7
    "",                                               # 8
    "Integrator complex",                             # 9
    "Neddylation",                                    # 10
    "RNA degradation",                                # 11
    "DNA Methylation",                                # 12
    "SIN3 complex",                                   # 13
    "",                                               # 14
    "DNA Methylation",                                # 15
    "",                                               # 16
    "snRNA Transcription; Cajal Body; SMN complex",   # 17
    "TGF-beta Signaling",                             # 18
    "CCR4-NOT complex",                               # 19
    "POU domain",                                     # 20
    "Transcription Regulation",                       # 21
    "",                                               # 22
    "SAGA complex",                                   # 23
    "Mediator complex",                               # 24
    "TFIID complex",                                  # 25
    "Mitochondrial Translation",                      # 26
    "Mitochondrial Transcription",                    # 27
    "Mitochondrial Membrane: Apoptosome",             # 28
    "Ragulator complex; Late Endosomes",              # 29
]
assert len(MANUAL_HDBSCAN) == 30, f'expected 30 hdbscan labels, got {len(MANUAL_HDBSCAN)}'

In [ ]:
# Point coords + cluster ids
mde = pd.read_excel(XLSX, sheet_name='MDE').rename(columns={
    'gene_target': 'gene',
    'leiden cluster': 'leiden',
    'hdbscan cluster': 'hdbscan',
})
mde.head()

In [ ]:
# n_DEGs lookup (CSV is tab-separated, KOLF column holds gene symbols)
ndegs = pd.read_csv(NDEGS, sep='\t', index_col=0)
kolf_ndegs = dict(zip(ndegs['KOLF'].astype(str),
                      pd.to_numeric(ndegs['Number of DEGs_KOLF'], errors='coerce')))
mde['n_degs'] = mde['gene'].map(kolf_ndegs).fillna(-1).astype(int)
print(f'{len(mde)} perts; {int((mde.n_degs < 0).sum())} missing n_DEGs')

In [ ]:
leiden_labels  = {str(i): MANUAL_LEIDEN[i]  for i in range(len(MANUAL_LEIDEN))}
hdbscan_labels = {str(i): MANUAL_HDBSCAN[i] for i in range(len(MANUAL_HDBSCAN))}
print(f"leiden  labeled: {sum(bool(v) for v in leiden_labels.values())}/{len(leiden_labels)}")
print(f"hdbscan labeled: {sum(bool(v) for v in hdbscan_labels.values())}/{len(hdbscan_labels)}")

In [ ]:
points = [
    {'g': r.gene, 'x': round(float(r.x), 3), 'y': round(float(r.y), 3),
     'l': int(r.leiden), 'h': int(r.hdbscan), 'n': int(r.n_degs)}
    for r in mde.itertuples(index=False)
]

payload = {
    'points': points,
    'leiden_labels': leiden_labels,
    'hdbscan_labels': hdbscan_labels,
}

OUT.parent.mkdir(parents=True, exist_ok=True)
with open(OUT, 'w') as f:
    json.dump(payload, f, separators=(',', ':'))

print(f'wrote {OUT} ({OUT.stat().st_size/1024:.1f} KB, {len(points)} points)')

## Preview locally

```bash
cd docs && python -m http.server 8000
# then open http://localhost:8000
```

## Deploy on GitHub Pages

1. Commit `docs/` and push to `main`.
2. GitHub → repo **Settings → Pages**: set **Source = Deploy from a branch**, **Branch = `main`**, **folder `/docs`**. Save.
3. Site is live at `https://y-doctor.github.io/KOLF2.1J_Perturbation_Cell_Atlas/`.